# C16_E03 — Detector de anomalías para mantenimiento predictivo

**Capítulo:** 16 — Machine Learning, Edge AI y supervisión inteligente de lazos  
**Identificador:** `C16_E03`  
**Objetivo pedagógico:** Mantenimiento predictivo: detectar anomalías a partir de telemetría.  
**Librerías:** scikit-learn, numpy, matplotlib

## Ejemplo industrial

Detección de anomalías en variadores de frecuencia o válvulas de control.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Telemetría de actuador con anomalía latente

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import os
np.random.seed(0)

# Operación nominal: 3 features (vibración, corriente, temperatura)
N = 1000
X_nom = np.random.normal(loc=[0.5, 10.0, 60.0], scale=[0.05, 0.5, 1.0], size=(N, 3))
# Inyección de anomalías al final
X_anom = np.random.normal(loc=[1.2, 12.0, 65.0], scale=[0.1, 0.5, 1.0], size=(50, 3))
X = np.vstack([X_nom, X_anom])

## 2. Entrenamiento y scoring

In [2]:
clf = IsolationForest(contamination=0.05, random_state=0).fit(X_nom)
scores = clf.decision_function(X)
preds = clf.predict(X)   # 1 = nominal, -1 = anomalía
print(f"Anomalías detectadas: {(preds == -1).sum()} de {(np.r_[np.ones(N), -np.ones(50)] == -1).sum()} reales")

Anomalías detectadas: 100 de 50 reales


## 3. Visualización de scores

In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scores, label="score (IsolationForest)")
ax.axvline(N, color='red', alpha=0.3, label="Inicio anomalías")
ax.set_xlabel("muestra"); ax.set_ylabel("score")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C16_E03 — Detección de anomalías por IsolationForest")
out = '../../figures/cap16/fig_C16_F02_anomalias.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 4. Interpretación

IsolationForest aísla puntos anómalos midiendo la profundidad media en árboles aleatorios. Score bajo ⇒ punto anómalo. Aplicación industrial: detectar anomalías en variadores, motores, válvulas a partir de telemetría. **Métricas honestas obligatorias** en cualquier despliegue: tasa de falsos positivos (cuántas alarmas innecesarias) y tasa de falsos negativos (cuántos fallos no detectados). Sin ellas, la oferta es marketing.